            # Deploy google/medgemma-27b-text-it to EC2 with vLLM and benchmark across instance families

            This notebook provisions Amazon EC2 instances, serves
            [MedGemma 27B](https://huggingface.co/google/medgemma-27b-text-it) with vLLM in a
            Docker container, then benchmarks each deployment with
            [LLMeter](https://github.com/awslabs/llmeter). At the end it
            compares throughput and cost across experiments.

            **Task under test:** extract a structured JSON booking record from each email. Inputs are loaded from
            `sample-data/travel/01-domestic-flight.jsonl` (synthesized by
            `sample-data/scripts/synthesize.py --domain travel`).

            > **Notes**
            > * Each row represents the **optimum packing** for MedGemma 27B on that instance — maximum model replicas per instance-hour.
            > * MedGemma-27B (BF16) weights are ~55 GiB. g6.12xl is limited to 1 replica via TP=4; g6e.12xl fits 2 replicas (each 2× L40S = 96 GiB).
            > * MedGemma-27B is built on the Gemma 3 architecture. vLLM's Neuron backend does not currently support Gemma 3, so inf2/trn1 are out of scope.
            > * **Default region is us-west-2 (PDX).** Alternates: `us-east-2` (CMH) and `us-east-1` (IAD).

            ### Prerequisites

            1. **AWS credentials** configured for the target account (profile `default` by default). The identity needs EC2 + IAM + SSM permissions.
            2. **A Hugging Face token** with access to [`google/medgemma-27b-text-it`](https://huggingface.co/google/medgemma-27b-text-it) (a gated model). The notebook stores it in AWS Secrets Manager.
3. **A default VPC** in the target region.

            Everything else — IAM role & instance profile, per-experiment
            security group, subnet discovery, AMI lookup, vLLM Docker setup,
            and teardown — is handled by `DeploymentRunner`.


## 0. Preparation

### 0.1 Install / verify Python dependencies

In [ ]:
%pip install -q -U \
    "boto3>=1.34" "botocore>=1.34" \
    "llmeter>=0.1.11" "openai>=1.50" \
    "plotly>=5.24" "ipywidgets>=8.1" \
    "huggingface_hub>=0.26" "pandas>=2.2" "matplotlib>=3.9" \
    "requests>=2.32" "tenacity>=9.0" "python-dotenv>=1.0" "jmespath>=1.0"


### 0.2 Imports and logging

In [ ]:
import asyncio
import json
import logging
import os
import sys
from datetime import datetime
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "models" / "medgemma_27b").is_dir():
    PROJECT_ROOT = NOTEBOOK_DIR
elif NOTEBOOK_DIR.name == "medgemma_27b":
    PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
else:
    raise RuntimeError(
        f"Can't locate project root from CWD={NOTEBOOK_DIR}. "
        "Launch Jupyter from the project root or from models/medgemma_27b/."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
_SRC = PROJECT_ROOT / "src"
if _SRC.is_dir() and str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import boto3
import pandas as pd

from vllm_ec2_bench import (
    DeploymentRunner,
    ExperimentConfig,
    catalog_meta,
    rates_from_responses,
    scrape_vllm_metrics,
    upsert_hf_token,
    verify_tier,
)
from vllm_ec2_bench.cleanup import (
    terminate_all_tagged_instances,
    cleanup_tagged_security_groups,
)
from vllm_ec2_bench.endpoint import (
    UniquePayloadEndpoint,
    VLLMEndpoint,
    make_http_client,
)
from models.medgemma_27b import (
    MEDGEMMA_27B,
    EXPERIMENTS,
    INSTANCE_TYPES,
    CATALOG_CACHE,
    DEFAULT_REGIONS,
    SYSTEM_PROMPT,
    SEED_INPUT,
    development_experiments,
    get as get_experiment,
    load_catalog,
    refresh_catalog,
)

from llmeter.runner import Runner

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)
LOG = logging.getLogger("notebook")
LOG.info("boto3=%s, pandas=%s", boto3.__version__, pd.__version__)


### 0.3 Region + Hugging Face token configuration

* `REGION` — default region for every experiment (`us-west-2`, PDX).
* `ALT_REGION_1` — first fallback (`us-east-2`, CMH).
* `ALT_REGION_2` — second fallback (`us-east-1`, IAD).
* `HF_TOKEN` — read access to `google/medgemma-27b-text-it` (gated repo).


In [ ]:
REGION = "us-west-2"
ALT_REGION_1 = "us-east-2"
ALT_REGION_2 = "us-east-1"

HF_TOKEN = "PLACEHOLDER_PASTE_YOUR_HF_TOKEN"
HF_SECRET_NAME = f"{MEDGEMMA_27B.resource_prefix}-benchmark/hf-token"

# Must cover sum(c * PER_CLIENT_REQUESTS) across all tiers, since each
# request consumes a DISTINCT input and the cursor advances between
# tiers. A c=800 sweep needs ~16,000; the pooled corpus holds 10,000,
# so the assert in the run helper will tell you if a plan outgrows it.
N_BENCHMARK_SAMPLES = 10000
N_WARMUP_SAMPLES = 5
BENCHMARK_SAMPLE_SEED = 42

assert 1 <= N_BENCHMARK_SAMPLES <= 100_000
assert 0 <= N_WARMUP_SAMPLES <= 1000

print(f"REGION              = {REGION}")
print(f"HF_SECRET_NAME      = {HF_SECRET_NAME}")
print(f"N_BENCHMARK_SAMPLES = {N_BENCHMARK_SAMPLES}")
print(f"N_WARMUP_SAMPLES    = {N_WARMUP_SAMPLES}")


**Upsert HF token into Secrets Manager.** Each EC2 instance fetches it via Secrets Manager at boot.

In [ ]:
assert HF_TOKEN != "PLACEHOLDER_PASTE_YOUR_HF_TOKEN", \
    "Scroll up and paste your HF token before running this cell."
assert HF_TOKEN.startswith("hf_"), \
    f"{HF_TOKEN[:8]}... doesn't look like an HF token."

secret_arn = upsert_hf_token(HF_SECRET_NAME, HF_TOKEN, region=REGION)
print(f"HF token stored in: {secret_arn}")
HF_TOKEN = "(stored in Secrets Manager)"


### 0.4 Preflight — AWS identity

In [ ]:
sts = boto3.client("sts")
identity = sts.get_caller_identity()
print(f"Account: {identity['Account']}")
print(f"ARN:     {identity['Arn']}")


### 0.5 Refresh the hardware catalog (pricing + specs from AWS APIs)

In [ ]:
CATALOG = load_catalog(offline_ok=False, max_age_hours_prices=24)
_meta = catalog_meta(CATALOG_CACHE)
print(f"Cache:             {CATALOG_CACHE}")
print(f"Catalog entries:   {len(CATALOG.instance_types())}")
print(f"Prices refreshed:  {_meta.get('prices_refreshed_at', '(just now)')}")
for it in INSTANCE_TYPES[:3]:
    hw = CATALOG.hardware(it)
    prices = CATALOG.price_od_all(it)
    if prices:
        region_prices = ", ".join(f"{r}=${p:.4f}" for r, p in sorted(prices.items()))
    else:
        region_prices = "(price unavailable)"
    print(f"  {it:<18} {hw.num_accelerators}× {hw.accelerator_model:<20} {region_prices}")


### 0.6 Load synthesized benchmark data

Benchmark inputs are pooled across every file in
`sample-data/travel/` (~10,000 synthesized travel booking confirmation emails), then
sampled deterministically.

Every request consumes one **distinct** input and the pool cursor
advances between concurrency tiers, so a large sweep needs
`sum(c * PER_CLIENT_REQUESTS)` inputs — several thousand for a
high-concurrency instance. Pooling all files rather than one keeps
that satisfiable; a single file holds only 1,000 records.


In [ ]:
import random

synth_dir = PROJECT_ROOT.parent / "sample-data" / "travel"
synth_files = sorted(synth_dir.glob("*.jsonl"))
assert synth_files, (
    f"No .jsonl files in {synth_dir}. Run "
    f"`python sample-data/scripts/synthesize.py --domain travel` first."
)

all_texts: list[str] = []
for synth_file in synth_files:
    with synth_file.open() as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            text = obj.get("text")
            if text:
                all_texts.append(text)

assert len(all_texts) >= N_BENCHMARK_SAMPLES, (
    f"Synthesis pool has only {len(all_texts)} records; "
    f"need at least N_BENCHMARK_SAMPLES={N_BENCHMARK_SAMPLES}"
)

rng = random.Random(BENCHMARK_SAMPLE_SEED)
INPUTS: list[str] = rng.sample(all_texts, N_BENCHMARK_SAMPLES)

print(f"Synthesis pool:    {len(all_texts):,} records "
      f"from {len(synth_files)} file(s)")
print(f"Sampled:           {len(INPUTS):,} prompts (seed={BENCHMARK_SAMPLE_SEED})")
print()
print("--- system prompt ---")
print(SYSTEM_PROMPT)
print()
print("--- first input (abbreviated) ---")
print(INPUTS[0][:320] + ("..." if len(INPUTS[0]) > 320 else ""))


In [ ]:
MAX_NEW_TOKENS = 512

# Requests per CLIENT per tier. Total requests scale WITH concurrency
# (c * PER_CLIENT_REQUESTS), which is what keeps every tier to a
# bounded, comparable duration. A flat per-tier total instead makes
# low-concurrency tiers run serially: at 40 s/request a 640-request
# c=1 tier takes 7 hours.
PER_CLIENT_REQUESTS = 10

def tier_requests(c: int) -> int:
    return c * PER_CLIENT_REQUESTS

# Tiers are derived from the PLAN's own concurrency_high so a large
# SKU is actually driven to saturation. Using one hardcoded ladder
# for every instance understates the big boxes badly: the same 8-GPU
# box measured $1.03/1M tokens at c=100 and $0.177/1M at c=800.
def tiers_for(exp_id: str) -> list[int]:
    high = get_experiment(exp_id).deployment.concurrency_high
    ladder = [1, 10, 30, 50, 100, 200, 400, 800, 1600]
    tiers = [c for c in ladder if c <= high]
    if high not in tiers:
        tiers.append(high)
    return tiers

# Client-side connection ceiling. The OpenAI SDK defaults to 1,000,
# which silently caps any tier above that and flattens the curve for
# a reason that has nothing to do with the GPU.
HTTP_MAX_CONNECTIONS = 4096

# Per-request timeout handed to LLMeter. Its default is 60 s and the
# per-client budget is timeout x n_requests; on expiry a client's
# entire response set is discarded while failed_requests still reads
# 0. Size this above the slowest expected single request.
REQUEST_TIMEOUT_S = 900

# Verification thresholds applied to every tier (see
# vllm_ec2_bench.verify). MAX_PREEMPTIONS=0 is the strict setting
# appropriate when you intend to quote the result: a tier where the
# engine evicted running sequences is not reproducible.
MIN_COMPLETENESS = 0.98
MAX_DIVERGENCE = 0.05
MAX_PREEMPTIONS = 0

print(f"Requests per client:     {PER_CLIENT_REQUESTS}")
print(f"Warmup requests:         {N_WARMUP_SAMPLES} at c=1 (discarded)")
print(f"Client conn ceiling:     {HTTP_MAX_CONNECTIONS}")
print(f"Per-request timeout:     {REQUEST_TIMEOUT_S}s")
print(f"Gates: completeness>={MIN_COMPLETENESS:.0%} "
      f"divergence<={MAX_DIVERGENCE:.0%} preemptions<={MAX_PREEMPTIONS}")


### 0.7 Shared helper to run one experiment end-to-end

In [ ]:
EXPERIMENTS_STATE: dict[str, dict] = {}

OUTPUT_BASE = Path("outputs")
OUTPUT_BASE.mkdir(exist_ok=True)


async def run_experiment(
    exp_id: str,
    *,
    concurrency_tiers: list[int],
    capacity_reservation_id: str | None = None,
    region_override: str | None = None,
) -> dict:
    cfg = get_experiment(exp_id)
    if region_override:
        new_plan = cfg.deployment.model_copy(update={"region": region_override})
        cfg = cfg.model_copy(update={"deployment": new_plan})

    hw = CATALOG.hardware(cfg.deployment.instance_type)
    print(f"[{exp_id}] spec: {cfg.deployment.instance_type} in {cfg.deployment.region} "
          f"(TP={cfg.deployment.tensor_parallel} DP={cfg.deployment.data_parallel})")
    print(f"[{exp_id}] total VRAM = {hw.vram_gib_total:.1f} GiB across "
          f"{hw.num_accelerators} accelerators")

    runner = DeploymentRunner(
        cfg,
        catalog=CATALOG,
        hf_secret_name=HF_SECRET_NAME,
        capacity_reservation_id=capacity_reservation_id,
    )
    state = runner.launch()

    # UniquePayloadEndpoint guarantees every request carries a
    # distinct input. Without it, LLMeter's constant-seeded shuffle
    # makes all clients replay the same handful of prompts, and
    # prefix caching turns those repeats into near-free cache hits
    # that inflate throughput. make_http_client lifts the OpenAI
    # SDK's 1,000-connection cap so tiers above c=1000 measure the
    # server rather than the client's connection pool.
    # The pool must cover the SUM of all tiers, because the cursor
    # advances between tiers rather than replaying the same inputs.
    # Replaying would let later tiers hit a prefix cache that earlier
    # tiers filled, inflating exactly the high-concurrency tiers that
    # decide the reported optimum.
    _needed_total = sum(tier_requests(c) for c in concurrency_tiers)
    assert len(INPUTS) >= _needed_total, (
        f"input pool {len(INPUTS)} < {_needed_total} required for "
        f"tiers {concurrency_tiers}; raise N_BENCHMARK_SAMPLES"
    )

    endpoint = UniquePayloadEndpoint(
        base_url=state.base_url,
        api_key=state.api_key,
        model_id=cfg.model_spec.served_model_name,
        inputs=INPUTS[:_needed_total],
        system_prompt=SYSTEM_PROMPT,
        max_tokens=MAX_NEW_TOKENS,
        http_client=make_http_client(HTTP_MAX_CONNECTIONS),
    )
    _pool = endpoint._client._client._transport._pool
    assert _pool._max_connections >= max(concurrency_tiers), (
        f"client pool {_pool._max_connections} < c="
        f"{max(concurrency_tiers)}; the sweep would be client-bound"
    )

    # Smoke test
    payload = VLLMEndpoint.create_payload(
        SYSTEM_PROMPT, INPUTS[0], max_tokens=MAX_NEW_TOKENS
    )
    smoke = endpoint.invoke(payload)
    print(f"[{exp_id}] smoke: "
          f"input_tokens={smoke.num_tokens_input} "
          f"output_tokens={smoke.num_tokens_output} "
          f"latency_s={smoke.time_to_last_token:.2f} "
          f"finish={endpoint.finish_reasons}")

    print(f"[{exp_id}] launch overhead (excluded): "
          f"spot_wait={state.capacity_wait_s or 0:.0f}s "
          f"vllm_warmup={state.vllm_ready_wait_s or 0:.0f}s")

    # One Runner per tier, not one LoadTest across all of them:
    #  * LoadTest does not expose `timeout`, and LLMeter's default is
    #    60 s per request. The per-client budget is timeout x n, and
    #    on expiry a client's ENTIRE response set is discarded while
    #    failed_requests still reads 0.
    #  * Each tier needs its own output directory. count_responses()
    #    globs recursively, so a shared directory makes every tier's
    #    verdict count every other tier's responses.
    #  * /metrics counters must be bracketed per tier; deltas across
    #    a whole sweep cannot attribute preemptions to a tier.
    import time as _time
    verdicts = {}
    tier_results = {}
    _cursor = 0
    for _c in concurrency_tiers:
        _needed = tier_requests(_c)
        _tier_dir = OUTPUT_BASE / exp_id / f"c{_c}"
        _tier_notes = INPUTS[_cursor:_cursor + _needed]
        endpoint.reset_pool(offset=_cursor)
        endpoint.reset_finish_reasons()
        _cursor += _needed

        # Placeholders: prepare_payload() swaps in a fresh input per
        # request, so only the LIST LENGTH matters here.
        _payloads = [
            VLLMEndpoint.create_payload(
                SYSTEM_PROMPT, x, max_tokens=MAX_NEW_TOKENS
            )
            for x in _tier_notes
        ]

        _m_before = scrape_vllm_metrics(state.base_url, state.api_key)
        _t0 = _time.time()
        _run = Runner(
            endpoint=endpoint,
            payload=_payloads,
            clients=_c,
            n_requests=PER_CLIENT_REQUESTS,
            output_path=str(_tier_dir),
            timeout=REQUEST_TIMEOUT_S,
            run_name=f"c{_c}",
            disable_per_client_progress_bar=True,
            disable_clients_progress_bar=True,
        )
        _res = await _run.run()
        _wall = _time.time() - _t0
        _m_after = scrape_vllm_metrics(state.base_url, state.api_key)
        _stats = getattr(_res, "stats", None) or {}

        # Rates recomputed from the response records over ONE window.
        # LLMeter divides input tokens by the first-to-last DISPATCH
        # window and output tokens by the dispatch-to-END window, so
        # summing its two rates adds figures measured over different
        # periods — a measured 9.4% inflation on a real tier, which
        # understates cost per token by the same margin.
        _rates = rates_from_responses(_tier_dir)
        _tpm = _rates.get("total_tokens_per_min") or 0
        _n_ok = _rates.get("n_successful") or 0
        _window = (
            _rates.get("window_s")
            or _stats.get("total_test_time")
            or _wall
        )
        _tot_tok = (_rates.get("total_input_tokens") or 0) + (
            _rates.get("total_output_tokens") or 0
        )

        _v = verify_tier(
            concurrency=_c,
            output_dir=_tier_dir,
            n_expected=_needed,
            stats_tokens_per_min=_tpm,
            total_tokens=_tot_tok,
            wall_clock_s=_window,
            metrics_before=_m_before,
            metrics_after=_m_after,
            min_completeness=MIN_COMPLETENESS,
            max_divergence=MAX_DIVERGENCE,
            max_preemptions=MAX_PREEMPTIONS,
        )

        # A truncated output means the item was never finished, so
        # any per-item cost derived from it is meaningless.
        _trunc = endpoint.truncated_count
        _done = endpoint.completed_count
        if _trunc:
            _v.valid = False
            _v.reasons.append(
                f"{_trunc}/{_trunc + _done} outputs hit "
                f"max_tokens={MAX_NEW_TOKENS} — items not completed; "
                "raise MAX_NEW_TOKENS and re-run"
            )

        verdicts[_c] = _v
        tier_results[_c] = {
            "stats": _stats,
            "rates": _rates,
            "responses_ok": _n_ok,
            "window_s": _window,
            "outputs_truncated": _trunc,
            "outputs_completed": _done,
            "wall_clock_s": _wall,
        }
        print(
            f"[{exp_id}] c={_c}: {_tpm:,.0f} tok/min, "
            f"{_n_ok}/{_needed} ok, trunc={_trunc} "
            f"[{'VALID' if _v.valid else 'INVALID'}]"
            + (f" — {'; '.join(_v.reasons)}" if _v.reasons else "")
        )

    _hit = [v.prefix_cache_hit_rate for v in verdicts.values()
            if v.prefix_cache_hit_rate is not None]
    if _hit:
        print(f"[{exp_id}] max prefix-cache hit rate: {max(_hit):.1%} "
              "(high values mean payload reuse, not a fast engine)")

    EXPERIMENTS_STATE[exp_id] = {
        "spec": cfg,
        "runner": runner,
        "state": state,
        "endpoint": endpoint,
        "tier_results": tier_results,
        "concurrency_tiers": concurrency_tiers,
        "capacity_wait_s": state.capacity_wait_s,
        "vllm_ready_wait_s": state.vllm_ready_wait_s,
        "verdicts": {k: v.as_dict() for k, v in verdicts.items()},
        "unique_inputs_served": endpoint.served,
        "completed_at": datetime.utcnow().isoformat(),
    }
    return EXPERIMENTS_STATE[exp_id]


def teardown_experiment(exp_id: str) -> None:
    entry = EXPERIMENTS_STATE.get(exp_id)
    if not entry:
        print(f"[{exp_id}] no state; nothing to tear down.")
        return
    runner: DeploymentRunner = entry["runner"]
    print(f"[{exp_id}] terminating {runner.state.instance_id} in {runner.state.region}...")
    runner.terminate()
    print(f"[{exp_id}] terminated.")


## Experiment 1 — g5.12xlarge (4× A10G / Ampere)

Capacity strategy: **spot (Fleet) → on-demand → auto-ODCR**.
The deployer walks each mode in order; the final mode is recorded
in `state.capacity_mode` and shown in the comparison table.

Concurrency tiers come from this experiment's own
`concurrency_high` via `tiers_for()` (section 0), so a large
instance is actually driven to saturation rather than swept with a
ladder borrowed from a smaller SKU.


In [ ]:
cfg = get_experiment("exp_1")
print(cfg.model_dump())
concurrency_tiers = tiers_for("exp_1")
print(f"tiers: {concurrency_tiers} "
      f"(requests: {[tier_requests(c) for c in concurrency_tiers]})")


In [ ]:
state_exp_1 = await run_experiment(
    "exp_1",
    concurrency_tiers=concurrency_tiers,
)


**Teardown for exp_1** — also deletes Spot Fleet, Launch Template, and any auto-created ODCR.

In [ ]:
# teardown_experiment("exp_1")  # <-- uncomment to tear down this experiment

## Experiment 2 — g6.12xlarge (4× L4 / Ada)

Capacity strategy: **spot (Fleet) → on-demand → auto-ODCR**.
The deployer walks each mode in order; the final mode is recorded
in `state.capacity_mode` and shown in the comparison table.

Concurrency tiers come from this experiment's own
`concurrency_high` via `tiers_for()` (section 0), so a large
instance is actually driven to saturation rather than swept with a
ladder borrowed from a smaller SKU.


In [ ]:
cfg = get_experiment("exp_2")
print(cfg.model_dump())
concurrency_tiers = tiers_for("exp_2")
print(f"tiers: {concurrency_tiers} "
      f"(requests: {[tier_requests(c) for c in concurrency_tiers]})")


In [ ]:
state_exp_2 = await run_experiment(
    "exp_2",
    concurrency_tiers=concurrency_tiers,
)


**Teardown for exp_2** — also deletes Spot Fleet, Launch Template, and any auto-created ODCR.

In [ ]:
# teardown_experiment("exp_2")  # <-- uncomment to tear down this experiment

## Experiment 3 — g6e.12xlarge (4× L40S / Ada)

Capacity strategy: **spot (Fleet) → on-demand → auto-ODCR**.
The deployer walks each mode in order; the final mode is recorded
in `state.capacity_mode` and shown in the comparison table.

Concurrency tiers come from this experiment's own
`concurrency_high` via `tiers_for()` (section 0), so a large
instance is actually driven to saturation rather than swept with a
ladder borrowed from a smaller SKU.


In [ ]:
cfg = get_experiment("exp_3")
print(cfg.model_dump())
concurrency_tiers = tiers_for("exp_3")
print(f"tiers: {concurrency_tiers} "
      f"(requests: {[tier_requests(c) for c in concurrency_tiers]})")


In [ ]:
state_exp_3 = await run_experiment(
    "exp_3",
    concurrency_tiers=concurrency_tiers,
)


**Teardown for exp_3** — also deletes Spot Fleet, Launch Template, and any auto-created ODCR.

In [ ]:
# teardown_experiment("exp_3")  # <-- uncomment to tear down this experiment

## Experiment 4 — g7e.2xlarge (1× Blackwell)

Capacity strategy: **spot (Fleet) → on-demand → auto-ODCR**.
The deployer walks each mode in order; the final mode is recorded
in `state.capacity_mode` and shown in the comparison table.

Concurrency tiers come from this experiment's own
`concurrency_high` via `tiers_for()` (section 0), so a large
instance is actually driven to saturation rather than swept with a
ladder borrowed from a smaller SKU.


In [ ]:
cfg = get_experiment("exp_4")
print(cfg.model_dump())
concurrency_tiers = tiers_for("exp_4")
print(f"tiers: {concurrency_tiers} "
      f"(requests: {[tier_requests(c) for c in concurrency_tiers]})")


In [ ]:
state_exp_4 = await run_experiment(
    "exp_4",
    concurrency_tiers=concurrency_tiers,
)


**Teardown for exp_4** — also deletes Spot Fleet, Launch Template, and any auto-created ODCR.

In [ ]:
# teardown_experiment("exp_4")  # <-- uncomment to tear down this experiment

## Experiment 5 — g7e.12xlarge (2× Blackwell)

Capacity strategy: **spot (Fleet) → on-demand → auto-ODCR**.
The deployer walks each mode in order; the final mode is recorded
in `state.capacity_mode` and shown in the comparison table.

Concurrency tiers come from this experiment's own
`concurrency_high` via `tiers_for()` (section 0), so a large
instance is actually driven to saturation rather than swept with a
ladder borrowed from a smaller SKU.


In [ ]:
cfg = get_experiment("exp_5")
print(cfg.model_dump())
concurrency_tiers = tiers_for("exp_5")
print(f"tiers: {concurrency_tiers} "
      f"(requests: {[tier_requests(c) for c in concurrency_tiers]})")


In [ ]:
state_exp_5 = await run_experiment(
    "exp_5",
    concurrency_tiers=concurrency_tiers,
)


**Teardown for exp_5** — also deletes Spot Fleet, Launch Template, and any auto-created ODCR.

In [ ]:
# teardown_experiment("exp_5")  # <-- uncomment to tear down this experiment

## Experiment 6 — p4d.24xlarge (8× A100 40GB / Ampere)

Capacity strategy: **spot (Fleet) → on-demand → auto-ODCR**.
The deployer walks each mode in order; the final mode is recorded
in `state.capacity_mode` and shown in the comparison table.

Concurrency tiers come from this experiment's own
`concurrency_high` via `tiers_for()` (section 0), so a large
instance is actually driven to saturation rather than swept with a
ladder borrowed from a smaller SKU.


In [ ]:
cfg = get_experiment("exp_6")
print(cfg.model_dump())
concurrency_tiers = tiers_for("exp_6")
print(f"tiers: {concurrency_tiers} "
      f"(requests: {[tier_requests(c) for c in concurrency_tiers]})")


In [ ]:
state_exp_6 = await run_experiment(
    "exp_6",
    concurrency_tiers=concurrency_tiers,
)


**Teardown for exp_6** — also deletes Spot Fleet, Launch Template, and any auto-created ODCR.

In [ ]:
# teardown_experiment("exp_6")  # <-- uncomment to tear down this experiment

## Experiment 7 — p4de.24xlarge (8× A100 80GB / Ampere)

Capacity strategy: **spot (Fleet) → on-demand → auto-ODCR**.
The deployer walks each mode in order; the final mode is recorded
in `state.capacity_mode` and shown in the comparison table.

Concurrency tiers come from this experiment's own
`concurrency_high` via `tiers_for()` (section 0), so a large
instance is actually driven to saturation rather than swept with a
ladder borrowed from a smaller SKU.


In [ ]:
cfg = get_experiment("exp_7")
print(cfg.model_dump())
concurrency_tiers = tiers_for("exp_7")
print(f"tiers: {concurrency_tiers} "
      f"(requests: {[tier_requests(c) for c in concurrency_tiers]})")


In [ ]:
state_exp_7 = await run_experiment(
    "exp_7",
    concurrency_tiers=concurrency_tiers,
)


**Teardown for exp_7** — also deletes Spot Fleet, Launch Template, and any auto-created ODCR.

In [ ]:
# teardown_experiment("exp_7")  # <-- uncomment to tear down this experiment

## Experiment 8 — p6-b200.48xlarge (8× B200 / Blackwell) — TP=1 DP=8, persistent spot wait

Capacity strategy: **spot (Fleet) → on-demand → auto-ODCR**.
The deployer walks each mode in order; the final mode is recorded
in `state.capacity_mode` and shown in the comparison table.

Concurrency tiers come from this experiment's own
`concurrency_high` via `tiers_for()` (section 0), so a large
instance is actually driven to saturation rather than swept with a
ladder borrowed from a smaller SKU.


In [ ]:
cfg = get_experiment("exp_8")
print(cfg.model_dump())
concurrency_tiers = tiers_for("exp_8")
print(f"tiers: {concurrency_tiers} "
      f"(requests: {[tier_requests(c) for c in concurrency_tiers]})")


In [ ]:
state_exp_8 = await run_experiment(
    "exp_8",
    concurrency_tiers=concurrency_tiers,
)


**Teardown for exp_8** — also deletes Spot Fleet, Launch Template, and any auto-created ODCR.

In [ ]:
# teardown_experiment("exp_8")  # <-- uncomment to tear down this experiment

## Performance and cost analysis

Per-tier throughput, $/1M-tokens, and percentile-latency comparison across experiments.

In [ ]:
STAT_VARIANT = "average"  # "p50" | "p90" | "p99"

def _get_per_tier_stats(entry: dict) -> dict[int, dict]:
    # Per-tier stats, with rate keys overridden by the recomputation.
    # The LLMeter average_*_tokens_per_minute pair uses two
    # different denominators, so anything derived from their sum is
    # inflated. rates_from_responses recomputes over a single
    # window; those values win wherever available.
    per_tier = {}
    for clients, tr in (entry.get("tier_results") or {}).items():
        stats = dict(tr.get("stats") or {})
        rates = tr.get("rates") or {}
        if rates.get("total_tokens_per_min"):
            stats["total_tokens_per_minute"] = rates["total_tokens_per_min"]
            stats["average_input_tokens_per_minute"] = rates[
                "input_tokens_per_min"
            ]
            stats["average_output_tokens_per_minute"] = rates[
                "output_tokens_per_min"
            ]
            stats["responses_ok"] = rates.get("n_successful")
            stats["measurement_window_s"] = rates.get("window_s")
        per_tier[int(clients)] = stats
    return per_tier


def build_comparison_df(stat_variant: str = "average") -> pd.DataFrame:
    rows: list[dict] = []
    all_tiers: set[int] = set()
    for entry in EXPERIMENTS_STATE.values():
        all_tiers.update(entry["concurrency_tiers"])
    all_tiers_sorted = sorted(all_tiers)

    for exp_id, entry in EXPERIMENTS_STATE.items():
        cfg = entry["spec"]
        dep = cfg.deployment
        hw = CATALOG.hardware(dep.instance_type)
        capacity_mode = entry["state"].capacity_mode
        od_price = CATALOG.price_od(dep.instance_type, dep.region)

        actual_hourly = od_price
        price_source = "OD"
        if capacity_mode == "spot":
            # Prefer the LIVE spot price. The 0.7×OD heuristic can be
            # wildly wrong for scarce accelerators — on p6-b200 it
            # reads $79.75/hr against an actual ~$40/hr, which
            # doubles every $/token figure derived from it. Fall back
            # to the heuristic only if the API call fails, and label
            # which one was used so the table never hides the basis.
            live = CATALOG.live_spot(dep.instance_type, dep.region)
            if live:
                actual_hourly = live
                price_source = "spot (live)"
            else:
                actual_hourly = CATALOG.estimated_spot(dep.instance_type, dep.region) or od_price
                price_source = "spot (estimated, 0.7×OD)*"

        # Launch overhead (capacity acquisition + vLLM warmup) is
        # tracked on the deployment state and is EXCLUDED from the
        # throughput/cost figures below: LLMeter measures tok/min
        # from the first request, after the endpoint is ready. For
        # scarce GPUs (p6-B200) the spot wait can be many minutes;
        # surfacing it here keeps it visible without contaminating
        # the per-token economics.
        dep_state = entry.get("state")
        cap_wait = getattr(dep_state, "capacity_wait_s", None)
        ready_wait = getattr(dep_state, "vllm_ready_wait_s", None)
        # Sum of the per-tier measurement windows.
        bench_wall = sum(
            (tr.get("wall_clock_s") or 0)
            for tr in (entry.get("tier_results") or {}).values()
        ) or None

        row = {
            "Experiment": exp_id,
            "Instance": dep.instance_type,
            "Region": dep.region,
            "GPUs": hw.num_accelerators,
            "GPU Model": hw.accelerator_model,
            "TP": dep.tensor_parallel,
            "DP": dep.data_parallel,
            "Replicas": cfg.model_replicas,
            "$/hr": round(actual_hourly, 4) if actual_hourly else None,
            "$/hr source": price_source,
            "Capacity": capacity_mode,
            "Spot wait (s)": round(cap_wait, 1) if cap_wait is not None else None,
            "vLLM warmup (s)": round(ready_wait, 1) if ready_wait is not None else None,
            "Benchmark run (s)": round(bench_wall, 1) if bench_wall is not None else None,
        }
        per_tier = _get_per_tier_stats(entry)
        verdicts = entry.get("verdicts") or {}
        for tier in all_tiers_sorted:
            stats = per_tier.get(tier, {})
            # Prefer the single-window recomputation; fall back to
            # LLMeter's sum only if it is unavailable, and mark it.
            total_tpm = stats.get("total_tokens_per_minute")
            basis = "recomputed"
            if not total_tpm:
                total_tpm = (
                    (stats.get("average_input_tokens_per_minute") or 0)
                    + (stats.get("average_output_tokens_per_minute") or 0)
                )
                basis = "llmeter (two-window; inflated)"
            cost_per_1m = None
            if total_tpm and actual_hourly:
                cost_per_1m = round(actual_hourly / (total_tpm * 60) * 1_000_000, 4)
            row[f"c={tier} tok/min"] = round(total_tpm, 1) if total_tpm else None
            row[f"c={tier} $/1M"] = cost_per_1m
            # Never let a gated-out tier be read as a clean result.
            v = verdicts.get(tier) or verdicts.get(str(tier)) or {}
            row[f"c={tier} valid"] = v.get("valid")
            if total_tpm:
                row[f"c={tier} rate basis"] = basis
        rows.append(row)
    return pd.DataFrame(rows) if rows else pd.DataFrame()

df_compare = build_comparison_df(stat_variant=STAT_VARIANT)
csv_path = OUTPUT_BASE / "comparison_table.csv"
df_compare.to_csv(csv_path, index=False)
print(f"Comparison table saved -> {csv_path}")
df_compare


## Cleanup

After benchmarking, make sure **every** instance is terminated. The
emergency sweep below terminates any running instance tagged
`Project={MEDGEMMA_27B.project_tag_value}` in all 3 regions.


In [ ]:
# for eid in list(EXPERIMENTS_STATE.keys()):
#     try:
#         teardown_experiment(eid)
#     except Exception as e:
#         print(f"[{eid}] teardown error: {e}")


In [ ]:
PROJECT_TAG = MEDGEMMA_27B.project_tag_value
for r in [REGION, ALT_REGION_1, ALT_REGION_2]:
    killed = terminate_all_tagged_instances(r, PROJECT_TAG)
    print(f"{r}: terminated {len(killed)} instance(s): {killed}")
    deleted = cleanup_tagged_security_groups(r, PROJECT_TAG)
    print(f"{r}: deleted {len(deleted)} security group(s): {deleted}")
